In [1]:
from pathlib import Path

# Works in a notebook (no __file__ exists) and later in a .py script (which does).
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent        # smart-city-traffic-capstone/
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
LOG_FILE = NOTEBOOK_DIR / "pipeline.log"

FIGURES_DIR.mkdir(exist_ok=True)

# Sanity check only, just for us while setting up, not part of the pipeline itself.
# From here on we'll use logging instead of print for anything that reports pipeline status.
print("Project root:", PROJECT_ROOT)
print("Data file found:", (DATA_DIR / "Metro_Interstate_Traffic_Volume.csv").exists())

Project root: C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone
Data file found: True


### Task 1: Data Pipeline Construction — Logging Setup (Logging Requirement 1)

Configures a logger via logging.getLogger(__name__) (never the root logger), with a console handler (INFO+) and a file handler (DEBUG+, writing to pipeline.log). This satisfies the requirement to configure the logger at the top of the script before any pipeline logic runs.

In [2]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

logger.info("Logging initialised. Writing detailed logs to %s", LOG_FILE)

2026-09-13 10:47:39 | INFO     | __main__ | Logging initialised. Writing detailed logs to C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone\part2_python\pipeline.log


### Task 1, Step 1: Load the Raw CSV

Loads the CSV with try/except around file I/O and parsing errors specifically (no bare except:), and logs an INFO message with the row and column counts on success, satisfying Logging Requirement 2.

In [3]:
import pandas as pd

def load_data(csv_path):
    """Load the raw traffic CSV into a DataFrame, with error handling for common failure modes."""
    logger.info("Loading data from %s", csv_path)
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        logger.error("Data file not found at %s", csv_path)
        raise
    except pd.errors.ParserError as e:
        logger.error("Failed to parse CSV file: %s", e)
        raise
    else:
        logger.info("Loaded %d rows and %d columns", df.shape[0], df.shape[1])
        return df

raw_df = load_data(DATA_DIR / "Metro_Interstate_Traffic_Volume.csv")
raw_df.head()

2026-09-13 11:11:28 | INFO     | __main__ | Loading data from C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone\data\Metro_Interstate_Traffic_Volume.csv
2026-09-13 11:11:28 | INFO     | __main__ | Loaded 48204 rows and 9 columns


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


### Task 1, Step 2: Validate the Data Schema

Before any cleaning or transformation, confirm every expected column is present. If any are missing, this logs an ERROR and raises immediately, since continuing with a malformed schema would make every downstream step unreliable. (The overall "log an ERROR and exit gracefully" requirement for the whole pipeline will be wired in once all steps exist, as an outer safety net around the full sequence, this raise is what that outer layer will catch.)

In [4]:
EXPECTED_COLUMNS = [
    "holiday", "temp", "rain_1h", "snow_1h", "clouds_all",
    "weather_main", "weather_description", "date_time", "traffic_volume",
]

def validate_schema(df, expected_columns=EXPECTED_COLUMNS):
    """Validate that all expected columns are present before any further processing."""
    missing = [col for col in expected_columns if col not in df.columns]
    if missing:
        logger.error("Schema validation failed. Missing columns: %s", missing)
        raise ValueError(f"Missing expected columns: {missing}")
    logger.info("Schema validation passed: all %d expected columns are present.", len(expected_columns))
    return True

validate_schema(raw_df)

2026-09-13 11:12:54 | INFO     | __main__ | Schema validation passed: all 9 expected columns are present.


True

### Task 1, Step 3a: Standardise Categorical Values — Inspection

Before applying any fix, inspect the three categorical/text columns (holiday, weather_main, weather_description) for inconsistencies such as stray whitespace, inconsistent casing, or placeholder text masquerading as a real value. This is exploratory, not part of the pipeline itself, so print is fine here.

In [5]:
print("holiday: unique values")
print(raw_df["holiday"].unique())
print()

print("weather_main: unique values")
print(sorted(raw_df["weather_main"].unique()))
print()

print(f"weather_description: {raw_df['weather_description'].nunique()} unique values")
print(sorted(raw_df["weather_description"].unique()))

holiday: unique values
[nan 'Columbus Day' 'Veterans Day' 'Thanksgiving Day' 'Christmas Day'
 'New Years Day' 'Washingtons Birthday' 'Memorial Day' 'Independence Day'
 'State Fair' 'Labor Day' 'Martin Luther King Jr Day']

weather_main: unique values
['Clear', 'Clouds', 'Drizzle', 'Fog', 'Haze', 'Mist', 'Rain', 'Smoke', 'Snow', 'Squall', 'Thunderstorm']

weather_description: 38 unique values
['SQUALLS', 'Sky is Clear', 'broken clouds', 'drizzle', 'few clouds', 'fog', 'freezing rain', 'haze', 'heavy intensity drizzle', 'heavy intensity rain', 'heavy snow', 'light intensity drizzle', 'light intensity shower rain', 'light rain', 'light rain and snow', 'light shower snow', 'light snow', 'mist', 'moderate rain', 'overcast clouds', 'proximity shower rain', 'proximity thunderstorm', 'proximity thunderstorm with drizzle', 'proximity thunderstorm with rain', 'scattered clouds', 'shower drizzle', 'shower snow', 'sky is clear', 'sleet', 'smoke', 'snow', 'thunderstorm', 'thunderstorm with drizzle'

### Task 1, Step 3a: Standardise Categorical Values — Fix

holiday is already correctly parsed as missing by pandas (confirmed, no fix needed). weather_main is clean (confirmed, no fix needed). weather_description has case-inconsistent duplicates ('Sky is Clear' vs 'sky is clear') and an all-caps outlier ('SQUALLS'); lower-casing the column merges these into consistent categories. Each finding is logged separately per the logging requirement, rows are only logged as changed when a value actually differs before and after.

In [6]:
def standardize_categoricals(df):
    """Standardise inconsistent categorical values before further processing."""
    df = df.copy()

    # holiday: confirm it's already correctly missing, not the literal string "None".
    non_holiday_count = df["holiday"].isna().sum()
    logger.info(
        "'holiday' already parsed as missing (NaN) for %d non-holiday rows "
        "(pandas recognised the source CSV's literal 'None' text automatically).",
        non_holiday_count,
    )

    # Defensive whitespace strip on both text columns.
    for col in ["weather_main", "weather_description"]:
        stripped = df[col].str.strip()
        changed = (stripped != df[col]).sum()
        if changed > 0:
            logger.warning("Stripped whitespace from %d rows in '%s'.", changed, col)
        df[col] = stripped

    # weather_description: standardise casing, merges case-variant duplicates
    # (e.g. 'Sky is Clear' / 'sky is clear') and fixes the all-caps 'SQUALLS'.
    before_unique = df["weather_description"].nunique()
    lowered = df["weather_description"].str.lower()
    changed = (lowered != df["weather_description"]).sum()
    if changed > 0:
        logger.warning(
            "Standardised casing to lowercase in %d rows of 'weather_description' "
            "(merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').",
            changed,
        )
    df["weather_description"] = lowered
    logger.info(
        "'weather_description' now has %d unique values (was %d before standardisation).",
        df["weather_description"].nunique(), before_unique,
    )

    return df

clean_df = standardize_categoricals(raw_df)
sorted(clean_df["weather_description"].unique())

2026-09-13 11:22:21 | INFO     | __main__ | 'holiday' already parsed as missing (NaN) for 48143 non-holiday rows (pandas recognised the source CSV's literal 'None' text automatically).
2026-09-13 11:22:21 | WARNING  | __main__ | Standardised casing to lowercase in 1730 rows of 'weather_description' (merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').
2026-09-13 11:22:21 | INFO     | __main__ | 'weather_description' now has 37 unique values (was 38 before standardisation).


['broken clouds',
 'drizzle',
 'few clouds',
 'fog',
 'freezing rain',
 'haze',
 'heavy intensity drizzle',
 'heavy intensity rain',
 'heavy snow',
 'light intensity drizzle',
 'light intensity shower rain',
 'light rain',
 'light rain and snow',
 'light shower snow',
 'light snow',
 'mist',
 'moderate rain',
 'overcast clouds',
 'proximity shower rain',
 'proximity thunderstorm',
 'proximity thunderstorm with drizzle',
 'proximity thunderstorm with rain',
 'scattered clouds',
 'shower drizzle',
 'shower snow',
 'sky is clear',
 'sleet',
 'smoke',
 'snow',
 'squalls',
 'thunderstorm',
 'thunderstorm with drizzle',
 'thunderstorm with heavy rain',
 'thunderstorm with light drizzle',
 'thunderstorm with light rain',
 'thunderstorm with rain',
 'very heavy rain']

### Task 1, Step 3b: Parse and Validate date_time

Converts date_time from its raw string form into a proper pandas datetime dtype using pd.to_datetime(..., errors="coerce"), so any value that fails to parse becomes a missing timestamp instead of silently crashing the pipeline or being misread. Any parse failures are logged as a WARNING with the affected row count; success is logged as INFO along with the resulting date range, which also serves as a sanity check against what we already know from Part 1 (should span 2012-10-02 to 2018-09-30).

In [7]:
def parse_and_validate_datetime(df, column="date_time"):
    """Parse the date/time column to a proper datetime dtype and validate the result."""
    df = df.copy()
    original_dtype = df[column].dtype

    parsed = pd.to_datetime(df[column], errors="coerce")

    failed_mask = parsed.isna() & df[column].notna()
    failed_count = failed_mask.sum()
    if failed_count > 0:
        logger.warning(
            "Failed to parse %d values in '%s' as valid datetimes; these rows now have a missing timestamp.",
            failed_count, column,
        )
    else:
        logger.info(
            "All %d values in '%s' parsed successfully as datetimes (was dtype '%s').",
            len(df), column, original_dtype,
        )

    df[column] = parsed
    logger.info("'%s' range: %s to %s.", column, df[column].min(), df[column].max())

    return df

clean_df = parse_and_validate_datetime(clean_df)
clean_df["date_time"].dtype, clean_df["date_time"].isna().sum()

2026-09-13 11:24:38 | INFO     | __main__ | All 48204 values in 'date_time' parsed successfully as datetimes (was dtype 'object').
2026-09-13 11:24:38 | INFO     | __main__ | 'date_time' range: 2012-10-02 09:00:00 to 2018-09-30 23:00:00.


(dtype('<M8[ns]'), np.int64(0))

### Task 1, Step 3c: Identify and Remove Duplicate Rows

Two distinct duplication issues can exist: exact full-row duplicates, and duplicate date_time entries where the weather feed logged more than one reading in the same hour (confirmed in Part 1: 5,445 such hours, with traffic_volume identical across the duplicates). We handle both: drop any exact duplicate rows first, then collapse duplicate timestamps down to one row per hour, verifying traffic_volume actually agrees across duplicates before collapsing rather than assuming it does.

In [8]:
def remove_duplicates(df, timestamp_col="date_time", value_col="traffic_volume"):
    """Identify and remove duplicate rows, including duplicate-timestamp entries."""
    df = df.copy()

    # 1. Exact full-row duplicates.
    exact_dupe_mask = df.duplicated()
    exact_dupe_count = exact_dupe_mask.sum()
    if exact_dupe_count > 0:
        logger.warning("Removed %d exact duplicate rows.", exact_dupe_count)
        df = df[~exact_dupe_mask]
    else:
        logger.info("No exact duplicate rows found.")

    # 2. Duplicate timestamps (same hour logged more than once by the weather feed).
    dupe_ts_mask = df.duplicated(subset=timestamp_col, keep=False)
    affected_timestamps = df.loc[dupe_ts_mask, timestamp_col].nunique()

    if affected_timestamps > 0:
        # Verify traffic_volume actually agrees within each duplicated hour before collapsing.
        inconsistent_groups = 0
        for ts, group in df.loc[dupe_ts_mask].groupby(timestamp_col):
            if group[value_col].nunique() > 1:
                inconsistent_groups += 1
                logger.warning(
                    "Duplicate timestamp %s has inconsistent '%s' values %s; keeping the first row anyway.",
                    ts, value_col, group[value_col].tolist(),
                )

        rows_before = len(df)
        df = df.drop_duplicates(subset=timestamp_col, keep="first")
        rows_removed = rows_before - len(df)

        logger.warning(
            "Collapsed %d duplicate-timestamp hours (%d rows removed) down to one row per hour; "
            "%d of these hours had inconsistent traffic_volume across duplicates.",
            affected_timestamps, rows_removed, inconsistent_groups,
        )
    else:
        logger.info("No duplicate timestamps found.")

    return df

clean_df = remove_duplicates(clean_df)
clean_df.shape

2026-09-13 11:28:52 | WARNING  | __main__ | Removed 17 exact duplicate rows.
2026-09-13 11:28:53 | WARNING  | __main__ | Collapsed 5430 duplicate-timestamp hours (7612 rows removed) down to one row per hour; 0 of these hours had inconsistent traffic_volume across duplicates.


(40575, 9)

### Task 1, Step 3d: Detect and Handle Outliers

Flags physically impossible readings, temp at or below 0 Kelvin (confirmed sensor error in Part 1), and rain_1h above 1000mm in a single hour (the real-world record for hourly rainfall is around 305mm, so anything past 1000mm is clearly a logging error, and this threshold still catches the dataset's known ~9,831mm anomaly). Flagged values are set to missing, then imputed using that specific calendar month's own median, computed and applied in an explicit loop rather than one global average, since a global median would be a poor stand-in for, say, a winter temperature reading.

In [9]:
import numpy as np

def detect_and_impute_outliers(df):
    """Detect physically impossible sensor readings and impute them using each
    calendar month's own median, rather than a single global average."""
    df = df.copy()
    df["month"] = df["date_time"].dt.month

    # Temperature: 0 Kelvin or below is physically impossible at this location.
    temp_invalid_mask = df["temp"] <= 0
    temp_invalid_count = temp_invalid_mask.sum()
    if temp_invalid_count > 0:
        logger.warning(
            "Found %d rows with physically impossible 'temp' readings (<= 0 Kelvin).",
            temp_invalid_count,
        )
        df.loc[temp_invalid_mask, "temp"] = np.nan

    # Rainfall: anything above 1000mm/hour is far beyond the most extreme hourly
    # rainfall ever recorded (~305mm), so treat it as a sensor/logging error.
    rain_invalid_mask = df["rain_1h"] > 1000
    rain_invalid_count = rain_invalid_mask.sum()
    if rain_invalid_count > 0:
        logger.warning(
            "Found %d rows with physically implausible 'rain_1h' readings (> 1000mm/hour): %s",
            rain_invalid_count, df.loc[rain_invalid_mask, "rain_1h"].tolist(),
        )
        df.loc[rain_invalid_mask, "rain_1h"] = np.nan

    # Impute both columns using each calendar month's own median.
    for column in ["temp", "rain_1h"]:
        for month in sorted(df["month"].unique()):
            month_mask = df["month"] == month
            missing_mask = month_mask & df[column].isna()
            missing_count = missing_mask.sum()

            if missing_count > 0:
                month_median = df.loc[month_mask, column].median()
                df.loc[missing_mask, column] = month_median
                logger.warning(
                    "Imputed %d missing '%s' value(s) in month %d using that month's median (%.2f).",
                    missing_count, column, month, month_median,
                )

    df = df.drop(columns="month")
    return df

clean_df = detect_and_impute_outliers(clean_df)
clean_df["temp"].min(), clean_df["rain_1h"].max()

2026-09-13 11:32:48 | WARNING  | __main__ | Found 10 rows with physically impossible 'temp' readings (<= 0 Kelvin).
2026-09-13 11:32:48 | WARNING  | __main__ | Found 1 rows with physically implausible 'rain_1h' readings (> 1000mm/hour): [9831.3]
2026-09-13 11:32:48 | WARNING  | __main__ | Imputed 4 missing 'temp' value(s) in month 1 using that month's median (265.48).
2026-09-13 11:32:48 | WARNING  | __main__ | Imputed 6 missing 'temp' value(s) in month 2 using that month's median (265.89).
2026-09-13 11:32:48 | WARNING  | __main__ | Imputed 1 missing 'rain_1h' value(s) in month 7 using that month's median (0.00).


(243.39, 55.63)

### Task 1: Full Pipeline Orchestration (Logging Requirement 4)

Chains all five steps into a single run_pipeline() function representing what pipeline.py's main flow will look like. Wrapped in a top-level try/except Exception (not a bare except:), so any failure at any stage is caught here, logged as an ERROR with exc_info=True for a full traceback in the log file, and the program exits gracefully via sys.exit(1) rather than crashing with an unhandled error dumped to the console.

In [10]:
import sys

def run_pipeline(csv_path):
    """Run the full Task 1 cleaning pipeline end-to-end, with a top-level safety net
    that logs a full error trace and exits gracefully if any stage fails."""
    try:
        df = load_data(csv_path)
        validate_schema(df)
        df = standardize_categoricals(df)
        df = parse_and_validate_datetime(df)
        df = remove_duplicates(df)
        df = detect_and_impute_outliers(df)
    except Exception:
        logger.error("Pipeline failed and could not complete.", exc_info=True)
        sys.exit(1)
    else:
        logger.info("Pipeline completed successfully. Final shape: %d rows, %d columns.", *df.shape)
        return df

clean_df = run_pipeline(DATA_DIR / "Metro_Interstate_Traffic_Volume.csv")
clean_df.shape

2026-09-13 11:45:38 | INFO     | __main__ | Loading data from C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone\data\Metro_Interstate_Traffic_Volume.csv
2026-09-13 11:45:38 | INFO     | __main__ | Loaded 48204 rows and 9 columns
2026-09-13 11:45:38 | INFO     | __main__ | Schema validation passed: all 9 expected columns are present.
2026-09-13 11:45:38 | INFO     | __main__ | 'holiday' already parsed as missing (NaN) for 48143 non-holiday rows (pandas recognised the source CSV's literal 'None' text automatically).
2026-09-13 11:45:38 | WARNING  | __main__ | Standardised casing to lowercase in 1730 rows of 'weather_description' (merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').
2026-09-13 11:45:38 | INFO     | __main__ | 'weather_description' now has 37 unique values (was 38 before standardisation).
2026-09-13 11:45:38 | INFO     | __main__ | All 48204 values in 'date_time' parsed successfully as datetimes (was 

(40575, 9)

In [12]:
run_pipeline(DATA_DIR / "this_file_does_not_exist.csv")

2026-09-13 11:46:30 | INFO     | __main__ | Loading data from C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone\data\this_file_does_not_exist.csv
2026-09-13 11:46:30 | ERROR    | __main__ | Data file not found at C:\Users\user\OneDrive\Documents\AI ML and DS Course NUS\Capstone Project\smart-city-traffic-capstone\data\this_file_does_not_exist.csv
2026-09-13 11:46:30 | ERROR    | __main__ | Pipeline failed and could not complete.
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_32356\1086999980.py", line 7, in run_pipeline
    df = load_data(csv_path)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_32356\388496460.py", line 7, in load_data
    df = pd.read_csv(csv_path)
  File "C:\Users\user\anaconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "C:\Users\user\anaconda3\Lib\site-packages\pandas\io\parsers\readers.py", li

SystemExit: 1

C:\Users\user\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
